<a href="https://colab.research.google.com/github/ShaunGves/FlyRank-AI/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShaunGves/FlyRank-AI/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/ShaunGves/FlyRank-AI.git
%cd FlyRank-AI

from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("Token loaded successfully" if os.environ["HF_TOKEN"] else "Token missing")

import duckdb, pandas as pd
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("""
CREATE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN '{}'
);
""".format(os.environ["HF_TOKEN"]))

Cloning into 'FlyRank-AI'...
remote: Enumerating objects: 156, done.
remote: Counting objects: 100% (156/156), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 156 (delta 61), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (156/156), 1.89 MiB | 12.52 MiB/s, done.
Resolving deltas: 100% (61/61), done.
/content/FlyRank-AI
Token loaded successfully


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: FlyRank's starter model reports a random forest achieving Precision@50 of 0.740, substantially beating the baseline rule's 0.240. My methodology question: the label used (trend_direction == "down") is itself calculated from the current data window, not a future observed outcome — so I'd ask: does this validation still hold if the label were redefined as a genuinely future decline, rather than a current-window bucket? A model predicting a same-window label risks looking stronger than it would on a true forward-looking task.

Finding 2: The starter results used client-holdout validation, where whole clients are kept out of training. My methodology question: with only a subset of clients in the starter slice, I'd ask how many unique clients were in the holdout set — if it's a small number, the reported precision could vary a lot depending on which specific clients happened to land in the test split, rather than reflecting a stable, generalizable result.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

query_data = """
SELECT content_hash_id, gsc_impressions, gsc_avg_position, ga4_sessions, ga4_engaged_sessions, sessions_ai
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
"""
df = con.execute(query_data).df().dropna()
df["label"] = (df["sessions_ai"] > 0).astype(int)

X = df[["gsc_impressions", "gsc_avg_position", "ga4_sessions", "ga4_engaged_sessions"]]
y = df["label"]

# BEFORE: random split (Week 5 style)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model_before = LogisticRegression(max_iter=1000).fit(X_train, y_train)
auc_before = roc_auc_score(y_test, model_before.predict_proba(X_test)[:, 1])

# AFTER: grouped split by content_hash_id (honest split)
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["content_hash_id"]))
X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]

model_after = LogisticRegression(max_iter=1000).fit(X_train_g, y_train_g)
auc_after = roc_auc_score(y_test_g, model_after.predict_proba(X_test_g)[:, 1])

print("BEFORE (random split) AUC:", auc_before)
print("AFTER (grouped by content_hash_id) AUC:", auc_after)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

BEFORE (random split) AUC: 0.7818079008955033
AFTER (grouped by content_hash_id) AUC: 0.7833893989133486


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I re-confirm my final feature set (gsc_impressions, gsc_avg_position, ga4_sessions, ga4_engaged_sessions) contains no FlyRank product-computed flags (health_score, priority_score, action_type) and does not include sessions_ai itself, which is the source of my label. This matches the same leakage check performed in Week 3, now re-run against my final Week 5/6 feature set.

In [3]:
# Re-confirm: no product flags, no label-derived columns in features
feature_columns = list(X.columns)
print("Features used in model:", feature_columns)

excluded_check = ["health_score", "priority_score", "action_type", "sessions_ai"]
leaked = [col for col in excluded_check if col in feature_columns]
print("Any leaked/excluded columns found in features?:", leaked if leaked else "None found — clean")

Features used in model: ['gsc_impressions', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions']
Any leaked/excluded columns found in features?: None found — clean


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original (bold) claim: "The model shows which pages will get more AI traffic if reworked."

Rewritten (safe) claim: "The model's ranking is associated with pages that have historically shown AI referral sessions, based on observed search and engagement signals. This is a directional, decision-support signal for prioritizing review — it does not predict that reworking a specific page will cause an increase in AI traffic, since no causal experiment was run."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ Confirm ] Every section above is filled — markdown thinking AND the code that backs it
- [ Confirm ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ Confirm ] No client names, URLs, or private queries anywhere
- [ Confirm ] My claims use careful words: observed, measured, directional, decision-support
- [ Confirm ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.